In [0]:
-- Enable Liquid Clustering at table creation
USE CATALOG lakehouse_demo;

CREATE TABLE IF NOT EXISTS silver.customers_lc (
  surrogate_key  BIGINT GENERATED ALWAYS AS IDENTITY,
  customer_id    STRING,
  full_name      STRING,
  email          STRING,
  segment        STRING,
  country        STRING,
  valid_from     TIMESTAMP,
  valid_to       TIMESTAMP,
  is_current     BOOLEAN,
  _updated_at    TIMESTAMP
)
USING DELTA
CLUSTER BY (customer_id, country);
  -- replaces ZORDER BY

-- Change clustering columns later without rewriting the table
ALTER TABLE silver.customers_lc
CLUSTER BY (customer_id, segment);  -- impossible with Z-ORDER without full rewrite


/*==============================================================================
  DECISION RULE: Z-ORDER vs LIQUID CLUSTERING
  
  Situation                                    | Recommendation
  ---------------------------------------------|----------------------------------
  DBR < 13.3                                   | Z-ORDER
  DBR 13.3+, new table                         | Liquid Clustering
  Existing table, stable query patterns        | Z-ORDER is fine
  Query patterns change frequently             | Liquid Clustering
  Very large table, rewrite cost too high      | Liquid Clustering
  Table has static partitions (year/month)     | Z-ORDER within partitions
  
  Key Differences:
  - Z-ORDER: Requires full table rewrite to change columns, good for static patterns
  - Liquid Clustering: Dynamic, can change columns without rewrite, adapts over time
  
==============================================================================*/